# Aula 13C — Simulador Interativo: Routing, Orchestration e Utility

Este laboratório usa **evidência do TIL quando disponível** e cai para **DEMO** explicitamente sintético quando não houver medições versionadas.

> Pergunta: **qual arquitetura entrega valor suficiente com qualidade, custo, latência e risco aceitáveis?**


## 📘 Glossário Vivo — conceitos-chave da Aula 13C

Use o Glossário Vivo durante o laboratório. Os conceitos abaixo formam a linguagem necessária para interpretar as decisões do simulador:

**[Baseline](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#baseline) · [Model Routing](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#roteamento-de-modelos) · [Model Orchestration](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#orquestração-de-modelos) · [Quality Gate](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#quality-gate) · [Escalation Rate](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#taxa-de-escalonamento) · [Utility Function](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#função-de-utilidade) · [Compound AI System](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#sistema-composto-de-ia) · [Cost per Inference](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#custo-por-inferência) · [Trade-off](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#trade-off)**

### 🧭 Durante o laboratório...

| Quando você estiver pensando em... | Consulte |
|---|---|
| estabelecer uma referência simples para comparação | [Baseline](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#baseline) |
| escolher modelos diferentes conforme a tarefa | [Model Routing](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#roteamento-de-modelos) |
| combinar modelos ou etapas | [Model Orchestration](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#orquestração-de-modelos) |
| decidir quando aceitar ou escalar uma resposta | [Quality Gate](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#quality-gate) |
| entender quantos casos chegam à camada premium | [Escalation Rate](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#taxa-de-escalonamento) |
| equilibrar qualidade, custo e latência | [Utility Function](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#função-de-utilidade) |
| analisar vários componentes trabalhando juntos | [Compound AI System](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#sistema-composto-de-ia) |
| estimar o impacto econômico das chamadas | [Cost per Inference](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#custo-por-inferência) |
| aceitar ganhos em uma dimensão e perdas em outra | [Trade-off](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#trade-off) |

> Não memorize os termos isoladamente. Use os links quando o comportamento do simulador levantar uma dúvida conceitual.


## 1. Evidência antes da decisão

O simulador procura `til-model-evidence.csv`. Campos mínimos: `system`, `quality` (0–1), `cost_per_1000` e `latency_ms`. O contrato está em `data/model-evidence/README.md`.

**EVIDENCE** = medições versionadas. **DEMO** = proxies didáticos. Métrica simulada nunca deve ser apresentada como benchmark real.


### Estado de execução e evidência

Este notebook segue o padrão **headless-first** do TIL: a execução completa (`Run All`) não depende de interação humana. Os widgets são uma camada opcional e o cenário interativo só é calculado quando o aluno clica em **Simular cenário**.

O notebook foi validado em execução local headless e no Kaggle.

O experimento `EDU-ORCH-001` produziu o primeiro conjunto de evidências reais do TIL. Quando `data/model-evidence/til-model-evidence.csv` está disponível e válido, esta aula opera em modo **EVIDENCE**.

A comparação atualmente versionada utiliza **TF-IDF + Multinomial Naive Bayes** e **DistilBERT multilíngue**, medidos no mesmo conjunto de teste com `f1_macro` como métrica principal.

> O modo `EVIDENCE` não transforma automaticamente arquiteturas simuladas em benchmarks reais. Cascades, taxas de escalonamento e cenários compostos continuam sendo hipóteses de engenharia até serem medidos diretamente.


In [ ]:
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display, Markdown, clear_output
import ipywidgets as widgets

REQ={"system","quality","cost_per_1000","latency_ms"}
DEMO=pd.DataFrame([
["tfidf_naive_bayes",.82,.20,12,"demo"],
["distilbert",.92,.75,58,"demo"],
["llm_or_human_review",.96,4.00,650,"demo"]],
columns=["system","quality","cost_per_1000","latency_ms","evidence_status"])

def candidates():
    p=[Path("data/model-evidence/til-model-evidence.csv"),
       Path("../data/model-evidence/til-model-evidence.csv"),
       Path("../../data/model-evidence/til-model-evidence.csv")]
    k = Path("/kaggle/input")

    if k.exists():
        for dataset_dir in k.iterdir():
            if not dataset_dir.is_dir():
                continue

            candidate = dataset_dir / "til-model-evidence.csv"

            if candidate.exists():
                p.append(candidate)

    return p

def load_evidence():
    for p in candidates():
        if p.exists():
            d=pd.read_csv(p)
            if d.empty or not REQ.issubset(d.columns): continue
            for c in ["quality","cost_per_1000","latency_ms"]:
                d[c]=pd.to_numeric(d[c],errors="coerce")
            d=d.dropna(subset=list(REQ))
            d=d[d.quality.between(0,1)&d.cost_per_1000.ge(0)&d.latency_ms.ge(0)]
            if len(d)>=2: return d.reset_index(drop=True),"EVIDENCE",str(p)
    return DEMO.copy(),"DEMO","fallback sintético"

EVIDENCE,MODE,SOURCE=load_evidence()
display(Markdown(f"### Modo: **{MODE}**  \n`{SOURCE}`"))
display(EVIDENCE)


## 2. Sistema composto

As linhas carregadas são candidatos `single`. O `cascade` usa a camada de menor custo como inicial e a de maior qualidade como premium. O **quality gate** controla uma **taxa de escalonamento simulada**.

Mesmo em modo EVIDENCE, o ponto intermediário do cascade é uma hipótese de engenharia até ser medido diretamente.

📚 Glossário: [Quality Gate](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#quality-gate) · [Escalation Rate](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#taxa-de-escalonamento) · [Model Routing](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#roteamento-de-modelos) · [Trade-off](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#trade-off).


In [ ]:
def mm(s):
    a,b=s.min(),s.max()
    return pd.Series(np.zeros(len(s)),index=s.index) if np.isclose(a,b) else (s-a)/(b-a)

def cascade(g):
    cheap=EVIDENCE.sort_values(["cost_per_1000","latency_ms"]).iloc[0]
    premium=EVIDENCE.sort_values(["quality","cost_per_1000"],ascending=[False,True]).iloc[0]
    r=float(np.clip(.05+.90*g,0,1))
    return {
      "system":f"cascade:{cheap.system}→{premium.system}",
      "quality":(1-r)*cheap.quality+r*premium.quality,
      "cost_per_1000":cheap.cost_per_1000+r*premium.cost_per_1000,
      "latency_ms":cheap.latency_ms+r*premium.latency_ms,
      "escalation_rate":r,"cheap":cheap.system,"premium":premium.system}

def evaluate(wq,wc,wl,g):
    w=np.array([wq,wc,wl],float); w=np.ones(3) if np.isclose(w.sum(),0) else w; w/=w.sum()
    d=EVIDENCE[["system","quality","cost_per_1000","latency_ms"]].copy()
    d["architecture"]="single"; d["escalation_rate"]=0.
    x=cascade(g)
    row=pd.DataFrame([{**{k:x[k] for k in ["system","quality","cost_per_1000","latency_ms","escalation_rate"]},"architecture":"cascade"}])
    d=pd.concat([d,row],ignore_index=True)
    d["utility"]=w[0]*mm(d.quality)-w[1]*mm(d.cost_per_1000)-w[2]*mm(d.latency_ms)
    return d.sort_values("utility",ascending=False).reset_index(drop=True),x,w


## 3. 🎛️ Simulador

Os sliders mudam **prioridades**, não as medições. Ajuste qualidade, custo, latência e gate; observe ranking, escalonamento e o mapa custo × qualidade.

📚 Glossário: [Utility Function](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#função-de-utilidade) · [Cost per Inference](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#custo-por-inferência) · [Compound AI System](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#sistema-composto-de-ia).


In [ ]:
sty={"description_width":"140px"}
lay=widgets.Layout(width="95%")

q=widgets.IntSlider(value=60,min=0,max=100,step=5,description="Qualidade",style=sty,layout=lay,continuous_update=False)
c=widgets.IntSlider(value=25,min=0,max=100,step=5,description="Custo",style=sty,layout=lay,continuous_update=False)
l=widgets.IntSlider(value=15,min=0,max=100,step=5,description="Latência",style=sty,layout=lay,continuous_update=False)
g=widgets.FloatSlider(value=.45,min=0,max=1,step=.05,description="Quality gate",style=sty,layout=lay,continuous_update=False)

preset=widgets.ToggleButtons(options=[("Equilibrado","b"),("Qualidade","q"),("Custo","c"),("Latência","l")],value="b")
run_button=widgets.Button(description="Simular cenário",button_style="primary",icon="play")
out=widgets.Output()


def render(_=None):
    d,x,w=evaluate(q.value,c.value,l.value,g.value)
    with out:
        clear_output(wait=True)
        win=d.iloc[0]
        note="evidência versionada" if MODE=="EVIDENCE" else "proxies sintéticos"
        display(Markdown(
            f"**Vencedor do cenário:** `{win.system}` · dados: **{note}**  \n"
            f"**Cascade:** `{x['cheap']}` → `{x['premium']}` · escalonamento **{x['escalation_rate']:.0%}**"
        ))
        z=d[["system","architecture","quality","cost_per_1000","latency_ms","escalation_rate","utility"]].copy()
        z.insert(0,"rank",range(1,len(z)+1))
        display(z.round(4))

        fig,ax=plt.subplots(figsize=(8,5))
        for _,r in d.iterrows():
            ax.scatter(r.cost_per_1000,r.quality,s=90)
            ax.annotate(r.system,(r.cost_per_1000,r.quality),xytext=(5,5),textcoords="offset points")
        ax.set(xlabel="Custo / 1.000 inferências",ylabel="Qualidade",title=f"Custo × qualidade — {MODE}")
        ax.grid(alpha=.25)
        plt.show()
        plt.close(fig)

        fig,ax=plt.subplots(figsize=(8,4))
        ax.bar(d.system,d.utility)
        ax.axhline(0,lw=1)
        ax.set_ylabel("Utility")
        plt.xticks(rotation=25,ha="right")
        plt.show()
        plt.close(fig)


def choose(ch):
    if ch.get("name")!="value":
        return
    vals={"b":(60,25,15),"q":(85,10,5),"c":(40,50,10),"l":(40,10,50)}
    q.value,c.value,l.value=vals[ch["new"]]


preset.observe(choose,names="value")
run_button.on_click(render)

display(widgets.VBox([
    widgets.HTML(f"<b>Dados:</b> {MODE}<br>Ajuste os parâmetros e clique em <b>Simular cenário</b>."),
    preset,q,c,l,g,run_button,out
]))


## 4. Sensibilidade do gate

Varra o gate para procurar **retorno decrescente**: mais escalonamento pode elevar custo e latência sem ganho proporcional de qualidade.

📚 Glossário: **Quality Gate**, **Escalation Rate**, **Trade-off**.


In [ ]:
rows=[]
for gate in np.linspace(0,1,21):
    d,x,w=evaluate(60,25,15,gate)
    r=d[d.architecture.eq("cascade")].iloc[0]
    rows.append([gate,r.escalation_rate,r.quality,r.cost_per_1000,r.latency_ms,r.utility])
S=pd.DataFrame(rows,columns=["gate","escalation_rate","quality","cost_per_1000","latency_ms","utility"])
display(S.round(4))
for y,label in [("quality","Qualidade"),("cost_per_1000","Custo / 1.000")]:
    fig,ax=plt.subplots(figsize=(8,4)); ax.plot(S.gate,S[y],marker="o"); ax.set(xlabel="Quality gate",ylabel=label,title=f"Gate × {label}"); ax.grid(alpha=.25); plt.show()


## 5. Como sair de DEMO e chegar a EVIDENCE

`executar → medir qualidade → medir latência → medir/estimar custo → registrar proveniência → salvar til-model-evidence.csv`

Registre dataset/split, hardware, versão do modelo, amostra e data no relatório experimental. Não misture unidades de custo sem declarar conversão.

O próximo conjunto esperado é: **TF-IDF + Naive Bayes → DistilBERT → LLM/revisão**.


## 6. Desafio e síntese

Teste políticas de **qualidade primeiro**, **custo primeiro** e **latência primeiro**. Responda: existe melhor modelo universal ou **melhor sistema condicionado ao contexto**?

Sua conclusão deve citar pelo menos quatro conceitos do **Glossário Vivo** e separar **evidência medida**, **hipótese** e **proxy didático**.

`métrica → custo do erro → threshold/abstenção → quality gate → routing → orchestration → compound AI system → decisão baseada em evidências`


## 7. Evidência real ativada — EDU-ORCH-001

A Aula 13C agora possui evidência medida e versionada produzida pelo próprio TIL.

O experimento `EDU-ORCH-001 — Model evidence: baseline clássico vs Transformer` produziu a primeira comparação operacional entre:

```text
TF-IDF + Multinomial Naive Bayes
versus
DistilBERT multilingual
```

No mesmo conjunto de teste (`n = 3807`), foram observados aproximadamente:

| Métrica | TF-IDF + NB | DistilBERT |
|---|---:|---:|
| F1 macro | 0.9099 | 0.9300 |
| Accuracy | 0.9241 | 0.9406 |
| Latência média | 1.09 ms | 34.38 ms |
| Proxy de runtime / 1000 | 0.030 s | 52.00 s |

Os valores de runtime foram medidos em CPU e não devem ser generalizados para GPUs ou outros hardwares.

### O que muda pedagogicamente?

A pergunta deixa de ser apenas:

> “qual modelo tem maior qualidade?”

e passa a ser:

> “qual arquitetura entrega utilidade suficiente para determinada combinação de qualidade, latência, custo e risco?”

Isso conecta diretamente:

```text
Model
→ Selection
→ Routing
→ Orchestration
→ Utility
→ Compound AI System
```

### Próximo marco experimental

O próximo passo não é mais “ativar EVIDENCE”. É **ampliar a matriz de evidências**.

Novos experimentos poderão acrescentar, de forma reproduzível:

```text
novos modelos
+ outros hardwares
+ outros datasets
+ novas tarefas
+ custo monetário quando observável
+ cascades realmente medidas
+ LLMs
+ revisão humana
```

Cada nova linha deverá preservar o contrato de proveniência definido em `data/model-evidence/README.md`.
